# 01. Phân Tích Khám Phá Dữ Liệu (Exploratory Data Analysis - EDA)

Notebook này phân tích toàn diện cấu trúc bộ dữ liệu biển báo giao thông GTSRB: tổng số lượng ảnh, kiểm tra số lớp, phân bố mẫu từng lớp, kiểm tra mất cân bằng lớp (class imbalance), thống kê kích thước ảnh và hiển thị lưới hình ảnh mẫu đa dạng.

In [ ]:
import os
import sys
import cv2
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Thêm thư mục gốc dự án vào sys.path
sys.path.append('..')
from src.config import GTSRB_CLASSES, NUM_CLASSES, DATASET_DISTRIBUTION_PATH, RESULT_DIR
from src.data_loader import load_raw_dataset, find_dataset_dir

In [ ]:
# 1. Nạp dữ liệu và kiểm tra thống kê
dataset_dir, is_sample = find_dataset_dir()
print(f"Đường dẫn dataset: {dataset_dir}")
print(f"Trạng thái: {'SAMPLE DEMO DATASET' if is_sample else 'REAL GTSRB DATASET'}")

images, labels, stats = load_raw_dataset(dataset_dir=dataset_dir)
print(f"Tổng số lượng ảnh: {stats['total_images']}")
print(f"Tổng số lớp (classes): {stats['total_classes']}")

In [ ]:
# 2. Kiểm tra phân bố số lượng ảnh từng lớp (Class Imbalance)
class_counts = stats['class_counts']
df_counts = pd.DataFrame([
    {'Class_ID': cid, 'Count': count, 'Name_VI': GTSRB_CLASSES.get(cid, {}).get('vi')}
    for cid, count in class_counts.items()
])

plt.figure(figsize=(14, 6))
plt.bar(df_counts['Class_ID'], df_counts['Count'], color='#1E40AF')
plt.title('Phân Bố Số Lượng Ảnh Theo Lớp (GTSRB Class Distribution)', fontsize=14, fontweight='bold')
plt.xlabel('Class ID')
plt.ylabel('Số Lượng Ảnh')
plt.grid(axis='y', linestyle=':', alpha=0.7)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(DATASET_DISTRIBUTION_PATH, dpi=300, bbox_inches='tight')
plt.show()

print(f"Lớp có nhiều mẫu nhất: Class {df_counts.loc[df_counts['Count'].idxmax()]['Class_ID']} ({df_counts['Count'].max()} ảnh)")
print(f"Lớp có ít mẫu nhất: Class {df_counts.loc[df_counts['Count'].idxmin()]['Class_ID']} ({df_counts['Count'].min()} ảnh)")

In [ ]:
# 3. Thống kê kích thước ảnh gốc
heights = [img.shape[0] for img in images]
widths = [img.shape[1] for img in images]

print(f"Kích thước chiều cao (Height): Min={min(heights)}, Max={max(heights)}, Mean={np.mean(heights):.1f}")
print(f"Kích thước chiều rộng (Width): Min={min(widths)}, Max={max(widths)}, Mean={np.mean(widths):.1f}")

In [ ]:
# 4. Trực quan hóa hình ảnh mẫu đại diện cho các lớp khác nhau trên cùng 1 figure
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
sample_indices = np.random.choice(len(images), 15, replace=False)

for idx, ax in zip(sample_indices, axes.ravel()):
    ax.imshow(images[idx])
    cid = labels[idx]
    name_vi = GTSRB_CLASSES.get(cid, {}).get('vi', f'Class {cid}')
    ax.set_title(f"ID {cid}: {name_vi[:18]}", fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Nhận Xét Dữ Liệu (EDA Data Insights)

1. **Số lượng lớp & Quy mô:** Bộ dữ liệu bao gồm 43 lớp biển báo giao thông chuẩn GTSRB khác nhau với màu sắc, hình dạng (tròn, tam giác, bát giác, hình vuông) và mục đích sử dụng đa dạng.
2. **Hiện tượng mất cân bằng lớp (Class Imbalance):**
   - Phân bố số lượng mẫu giữa các lớp có sự chênh lệch rõ rệt (một số lớp biển báo phổ biến như biển hạn chế tốc độ 50km/h có số lượng ảnh lớn hơn đáng kể so với biển hạn chế 20km/h).
   - Do đó, trong quá trình đánh giá cần sử dụng chỉ số **Weighted F1-score** và **Macro F1-score** bên cạnh Accuracy để đánh giá công bằng hiệu năng trên các lớp có ít mẫu.
3. **Kích thước ảnh không đồng nhất:** Ảnh đầu vào gốc có độ phân giải thay đổi (từ 15x15 đến trên 250x250 pixels). Vì vậy, tất cả hình ảnh cần được **Resize thống nhất về 32x32 pixels** trước khi đưa vào mô hình.
4. **Nhu cầu Data Augmentation:** Cần áp dụng xoay nhẹ, dịch chuyển và thay đổi độ sáng trên tập **Training** để tăng số lượng và tính đa dạng cho các lớp có ít dữ liệu, đồng thời giúp mô hình tổng quát hóa tốt hơn.